# PipeDRAM: Reproducing the Paper Figures

This notebook walks through the reproduction of Figures 4 and 9-14 of the PipeDRAM paper
(MICRO 2026) and the area analysis. It lists the figure generators, checks dataset
provenance, prints the expected numerical results, and regenerates each figure.

Run it from the repository root after `./scripts/setup.sh`. By default it only inspects
files and replots from the included paper results; the paper-scale workload runs are
commented out and must be enabled explicitly.

## 1. Repository root

In [ ]:
from pathlib import Path
import json
import subprocess
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "figure_manifest.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "figure_manifest.json").exists(), "start Jupyter from the repository root"

manifest = json.loads((ROOT / "figure_manifest.json").read_text())
checksums = json.loads((ROOT / "input_checksums.json").read_text())
mode_file = ROOT / "generated/mode.txt"
print(f"Repository root: {ROOT}")
print(f"Paper figures:   {', '.join(manifest['figures'])}")
print(f"Last run mode:   {mode_file.read_text().strip() if mode_file.exists() else 'none'}")

## 2. Figure map

Figure 4 quantifies the data-transposition overhead of prior PUD systems. Figures 9-11
show microbenchmark throughput, application performance and energy, and CPU-normalized
energy efficiency. Figures 12-14 analyze mat utilization, the bit-precision/data-size
trade-off, and exposed carry propagation.

In [ ]:
rows = ["| Figure | Generator | Output |", "|---:|---|---|"]
for number, entry in manifest["figures"].items():
    rows.append(f"| {number} | `{entry['generator']}` | `{entry['output']}` |")
area_entry = manifest["area_analysis"]
rows.append(f"| Area | `{area_entry['generator']}` | `{area_entry['output']}` |")
display(Markdown("\n".join(rows)))

## 3. Datasets

- **kmeans**: generated with the included Rodinia generator and seed 11. The seed is
  fixed because the input values determine the number of clustering iterations, and
  therefore the number of instrumented PUD operations.
- **heartwall**: uses the paper-scale `test_4k.avi`, which is not redistributed. A
  supplied copy must match the recorded SHA-256; otherwise the included paper results
  are used.
- **x264**: SPEC CPU2017 source and media are not redistributed; only its results are
  included.

See `docs/DATASETS.md` for details.

In [ ]:
for name, record in checksums.items():
    print(name)
    for key, value in record.items():
        print(f"  {key}: {value}")

Uncomment the next cell to generate the kmeans input (about 558 MB). The script checks the
object count and the SHA-256.

In [ ]:
# subprocess.run([ROOT / "scripts/generate_kmeans_input.sh"], check=True)

## 4. Expected numerical results

`expected_metrics.json` holds the reference values. A reproduction run writes the
recomputed values to `generated/metrics.json`, and `scripts/validate.sh` compares the two.

In [ ]:
expected = json.loads((ROOT / "expected_metrics.json").read_text())
actual_path = ROOT / "generated/metrics.json"
actual = json.loads(actual_path.read_text()) if actual_path.exists() else None

print(f"Workloads: {expected['workload_count']}")
print(f"x264 results present: {expected['x264_frozen_result_present']}")
print("\nNormalized geometric means (to SIMDRAM, compute only)"
      + ("; '=' marks a match with generated/metrics.json" if actual else ""))
for key, value in expected["normalized_geomeans"].items():
    match = actual and abs(actual["normalized_geomeans"][key] - value) < 1e-9
    print(f"  {'=' if match else ' '} {key:42s} {value:12.6f}")

In [ ]:
print("Figure 12: mean mat utilization")
for workload, values in expected["figure12_mat_utilization"].items():
    print(f"  {workload:5s}: {values['mean_percent']:6.2f}% ({values['mats_used']:2d}/32 mats used)")

print("\nFigure 13: multiplication crossover point (data elements)")
for precision, elements in sorted(expected["figure13_crossover_elements"].items(), key=lambda kv: int(kv[0])):
    print(f"  {precision:>2}-bit: {elements:,.0f}")

print("\nFigure 14: exposed carry-propagation overhead")
for workload, percent in expected["figure14_exposed_carry_percent"].items():
    print(f"  {workload:5s}: {percent:6.2f}%")

## 5. Area analysis

PipeDRAM's DRAM-side area overhead is 1.86% of the DRAM die: 1.11% for the MIMDRAM
fine-grained mat substrate, 0.60% for LISA inter-subarray isolation, and 0.15% for SALP
row-decoder latches. The memory controller provisions a 16 KiB schedule scratchpad; the
largest schedule has 1,350 four-byte entries (5,400 bytes). The CACTI estimate for the
extended controller is 0.27 mm^2 (0.05% of the evaluated CPU die).

The component and schedule values come from prior work and CACTI and are not recomputed
from first principles. The scripts recompute the sums, synthesize MABM with Yosys, and
check the permutation/inverse round trip with Icarus Verilog. See `docs/AREA_ANALYSIS.md`.

In [ ]:
area = json.loads((ROOT / "area/area_metrics.json").read_text())
area_result_path = ROOT / "generated/area_metrics.json"
area_result = json.loads(area_result_path.read_text()) if area_result_path.exists() else None

components = area["dram_components_percent"]
schedule_bytes = area["largest_schedule_entries"] * area["bytes_per_schedule_entry"]

print("DRAM-side area overhead")
for name, percent in components.items():
    print(f"  {name:30s} {percent:5.2f}%")
print(f"  {'Total':30s} {sum(components.values()):5.2f}%")
print(f"\nLargest schedule:       {schedule_bytes:,} bytes")
print(f"Provisioned scratchpad: {area['schedule_scratchpad_kib']} KiB")
print(f"Controller area:        {area['controller_area_mm2']:.2f} mm^2")
print(f"CPU-die overhead:       {area['cpu_die_area_percent']:.2f}%")

if area_result:
    print("\nRTL checks")
    print(f"  Yosys MABM logic cells: {area_result['mabm_logic_cells']}")
    print(f"  Icarus round-trip test: {area_result['round_trip_test']}")
else:
    print("\nRTL checks have not been run yet (scripts/reproduce_area.sh).")

In [ ]:
# subprocess.run([ROOT / "scripts/reproduce_area.sh"], cwd=ROOT, check=True)

## 6. Full reproduction

`quick` regenerates every figure from the included paper results (under two minutes).
`full` rebuilds and runs all redistributable workloads at paper scale (up to 24 hours,
64 GB RAM).

In [ ]:
# subprocess.run([ROOT / "scripts/setup.sh"], cwd=ROOT, check=True)

# subprocess.run([ROOT / "scripts/reproduce.sh", "quick"], cwd=ROOT, check=True)
# subprocess.run([ROOT / "scripts/validate.sh", "quick"], cwd=ROOT, check=True)

# subprocess.run([ROOT / "scripts/reproduce.sh", "full"], cwd=ROOT, check=True)
# subprocess.run([ROOT / "scripts/validate.sh", "full"], cwd=ROOT, check=True)

## 7. Validation

Validation checks the numerical results, the regenerated workload CSVs (full mode only),
the figure PDFs and their axis labels, the 150-DPI renderings against
`reference/renderings/`, and the checksums of the reference files.

In [ ]:
mode_path = ROOT / "generated/mode.txt"
if not mode_path.exists():
    print("No reproduction has been run yet.")
elif not (ROOT / ".venv/bin/python").exists():
    print("Run scripts/setup.sh before validating.")
else:
    mode = mode_path.read_text().strip()
    result = subprocess.run([ROOT / "scripts/validate.sh", mode], cwd=ROOT,
                            text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("validation failed")

## 8. Individual figures

`replot` uses the included paper results. `full` regenerates the workload CSVs that
Figures 4, 10, and 11 depend on (once per notebook session) before plotting. Figures 9
and 12-14 run the simulator directly in both modes.

Set `HEARTWALL_INPUT` to the verified video to rerun heartwall in `full` mode.

In [ ]:
FIGURE_MODE = "replot"  # or "full"
HEARTWALL_INPUT = None   # e.g., Path("/data/test_4k.avi")

assert FIGURE_MODE in {"replot", "full"}
render_dir = ROOT / "generated/notebook_renderings"
render_dir.mkdir(parents=True, exist_ok=True)
_full_results_ready = False

def run_figure(number):
    global _full_results_ready
    key = str(number).lower()
    workload_figures = {"4", "10", "11"}

    if FIGURE_MODE == "full" and key in workload_figures and not _full_results_ready:
        command = [ROOT / "scripts/prepare_results.sh"]
        if HEARTWALL_INPUT is not None:
            command += ["--heartwall-input", str(HEARTWALL_INPUT)]
        subprocess.run(command, cwd=ROOT, check=True)
        _full_results_ready = True

    command = [ROOT / "scripts/reproduce_figure.sh", key, FIGURE_MODE]
    if FIGURE_MODE == "full" and key in workload_figures:
        command.append("--reuse-results")
    subprocess.run(command, cwd=ROOT, check=True)

    entry = manifest["area_analysis"] if key == "area" else manifest["figures"][key]
    pdf = ROOT / "generated/figures" / entry["output"]
    prefix = render_dir / f"figure_{key}"
    subprocess.run(
        ["pdftoppm", "-f", "1", "-singlefile", "-r", "120", "-png", pdf, prefix],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    display(Image(filename=str(prefix) + ".png"))

### Figure 4: Data-transposition overhead

Simulates the 16 bit-serial microbenchmarks and overlays the 12 applications.

In [ ]:
run_figure("4")

### Figure 9: Microbenchmark throughput

Simulates 16 bit-serial and 4 bit-parallel operations at 4-, 8-, 16-, and 32-bit precision.

In [ ]:
run_figure("9")

### Figure 10: Application execution time and energy

Execution time and energy of the 12 applications, normalized to SIMDRAM.

In [ ]:
run_figure("10")

### Figure 11: CPU-normalized energy efficiency

Combines the PUD results with the measured CPU (RAPL) and GPU energy. The CPU and GPU
measurements are platform-dependent and are not remeasured in `full` mode.

In [ ]:
run_figure("11")

### Figure 12: Mat and chip utilization

Builds the dependency graph of each operation and folds the per-bit-position work onto
the 32 mats of a chip, for each application's operation mix.

In [ ]:
run_figure("12")

### Figure 13: Bit-precision and data-size trade-off

Evaluates the multiplication cost model over bit-precision and number of data elements
and draws the PipeDRAM/SIMDRAM crossover.

In [ ]:
run_figure("13")

### Figure 14: Exposed carry propagation

Sweeps the number of in-flight batches and computes the exposed hierarchical carry
overhead for linear, logarithmic, and quadratic operation classes.

In [ ]:
run_figure("14")

### Area: DRAM area overhead and MABM RTL

Recomputes the DRAM-side area sum and schedule size, synthesizes MABM with Yosys, runs
the Icarus Verilog round-trip test, and plots the area comparison.

In [ ]:
run_figure("area")